# Testing adaptivity with humanoidmaze-giant

In [1]:
import glob
import json
import os

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from agents import agents
from agents.goal_proposer import GCFlowGoalProposerAgent
from utils.datasets import Dataset, GCDataset, HGCDataset, CGCDataset
from utils.flax_utils import restore_agent
from wrappers.datafuncs_utils import make_env_and_datasets, to_oracle_reps

##=========== PATHS ===========##

DQC_PATH = '../../scratch/dqc-reproduce/sd100001s_33415523.0.33415522.1.20260415_020458/'
DQC_CKPT_NUM = 1000000

FLOW_PATH = '../../scratch/checkpoints/gc_flow_goal_proposer/observation_horizon_h1_100'
FLOW_CKPT_NUM = 5000000

DATASET_DIR = '../../scratch/data/humanoidmaze-giant-navigate-v0/humanoidmaze-giant-navigate-100m-v0/'
ENV_NAME = 'humanoidmaze-giant-navigate-v0'

datasets = sorted([f for f in glob.glob(f'{DATASET_DIR}/*.npz') if '-val.npz' not in f])
print(f'Found {len(datasets)} dataset shards')

/home/jennifer/miniconda3/envs/aorl/lib/python3.10/site-packages/Cython/Distutils/old_build_ext.py:15: DeprecationWarning: dep_util is Deprecated. Use functions from setuptools instead.
  from distutils.dep_util import newer, newer_group
/home/jennifer/miniconda3/envs/aorl/lib/python3.10/site-packages/Cython/Distutils/old_build_ext.py:15: DeprecationWarning: dep_util is Deprecated. Use functions from setuptools instead.
  from distutils.dep_util import newer, newer_group
<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead


Found 25 dataset shards


In [2]:
##=========== LOAD DQC AGENT ===========##

flags_path = os.path.join(DQC_PATH, 'flags.json')
with open(flags_path, 'r') as f:
    saved_flags = json.load(f)

agent_config = saved_flags['agent']

dataset_class_name = agent_config.get('dataset_class', 'GCDataset')
dataset_class = {'GCDataset': GCDataset, 'HGCDataset': HGCDataset, 'CGCDataset': CGCDataset}[dataset_class_name]

# placeholder flags required by the agent constructor
agent_config['actor_p_curgoal'] = 0.5
agent_config['actor_p_trajgoal'] = 0.5
agent_config['actor_p_randomgoal'] = 0.0
agent_config['actor_geom_sample'] = 0.5
agent_config['subgoal_steps'] = 100
agent_config['train_goal_proposer'] = False

dataset_npz = np.load(datasets[0])
train_dataset = dataset_class(Dataset.create(**dict(dataset_npz)), config=agent_config)

example_batch = train_dataset.sample(1)
dqc_agent = agents[agent_config['agent_name']].create(0, example_batch, agent_config)
dqc_agent = restore_agent(dqc_agent, DQC_PATH, DQC_CKPT_NUM)
print(f'Restored dqc_agent from {DQC_PATH} checkpoint {DQC_CKPT_NUM}')

##=========== LOAD SUBGOAL PROPOSER ===========##

proposer_config = dict(
    env_name=ENV_NAME,
    observations_key='oracle_reps',
    goal_key='actor_goals',
    actions_key='low_actor_goals',
    hidden_dims=(256, 256, 256),
    layer_norm=True,
    lr=3e-4,
    batch_size=256,
    value_p_curgoal=0.0,
    value_p_trajgoal=1.0,
    value_p_randomgoal=0.0,
    value_geom_sample=False,
    actor_p_curgoal=0.0,
    actor_p_trajgoal=1.0,
    actor_p_randomgoal=0.0,
    actor_geom_sample=True,
    gc_negative=False,
    subgoal_steps=100,
    discount=0.999,
    flow_steps=10,
    backup_horizon=25,
    goal_conditioned=False,
    horizon_conditioned=True,
)

env, base_train_dataset, val_dataset = make_env_and_datasets(
    ENV_NAME,
    dataset_path=datasets[0],
    use_oracle_reps=True,
)
proposer_train_dataset = CGCDataset(base_train_dataset, config=proposer_config)
proposer_example_batch = proposer_train_dataset.sample(1)

flow_agent = GCFlowGoalProposerAgent.create(proposer_example_batch, proposer_config)
flow_agent = restore_agent(flow_agent, FLOW_PATH, FLOW_CKPT_NUM)
print(f'Restored flow_agent from {FLOW_PATH} checkpoint {FLOW_CKPT_NUM}')

Restored from ../../scratch/dqc-reproduce/sd100001s_33415523.0.33415522.1.20260415_020458//params_1000000.pkl
Restored dqc_agent from ../../scratch/dqc-reproduce/sd100001s_33415523.0.33415522.1.20260415_020458/ checkpoint 1000000
Restored from ../../scratch/checkpoints/gc_flow_goal_proposer/observation_horizon_h1_100/params_5000000.pkl
Restored flow_agent from ../../scratch/checkpoints/gc_flow_goal_proposer/observation_horizon_h1_100 checkpoint 5000000


In [ ]:
from collections import defaultdict

class CountBasedNovelty:
    """Visit-count bonus: 1 / sqrt(1 + N(s)) on a regular grid."""

    def __init__(self, bin_size: float = 1.0):
        self.bin_size = bin_size
        self._counts: dict[tuple, int] = defaultdict(int)

    def _key(self, xy: np.ndarray) -> tuple:
        return tuple(np.floor(np.asarray(xy[:2]) / self.bin_size).astype(int).tolist())

    def score(self, candidates: np.ndarray) -> np.ndarray:
        return np.array([1.0 / np.sqrt(1.0 + self._counts[self._key(c)]) for c in candidates])

    def update(self, visited: np.ndarray) -> None:
        self._counts[self._key(visited)] += 1

    @property
    def num_unique_cells(self) -> int:
        return sum(1 for v in self._counts.values() if v > 0)

In [ ]:
def sigmoid(x):
    x = np.asarray(x)
    return np.where(x >= 0, 1 / (1 + np.exp(-x)), np.exp(x) / (1 + np.exp(x)))


def sample_subgoals(ob, n, sample_rng):
    """Sample n subgoal candidates from the flow proposer given a raw observation."""
    oracle_rep = np.asarray(to_oracle_reps(obs=np.asarray(ob)[None], env=env))[0]
    obs_batch = np.repeat(oracle_rep[None], n, axis=0)
    return np.asarray(flow_agent.sample_actions(obs_batch, None, sample_rng))


def select_subgoal_with_novelty(
    ob,
    goal,
    novelty_scorer,
    rng,
    num_candidates: int = 128,
    mult_factor: float = 0.9,
    additive_factor: float = 0.0,
):
    """Sample candidates, filter by DQC improvement, score by novelty.

    Returns (subgoal, candidates, novelty_scores, mask).
    Falls back to pure novelty (no DQC filter) if no candidate clears the bar.
    """
    rng, sample_rng = jax.random.split(rng)
    candidates = sample_subgoals(ob, num_candidates, sample_rng)

    oracle_rep = np.asarray(to_oracle_reps(obs=np.asarray(ob)[None], env=env))[0]

    all_obs = np.repeat(oracle_rep[None], len(candidates), axis=0)
    ob_to_subgoal_vs = dqc_agent.network.select('value')(all_obs, candidates)
    subgoal_to_goal_vs = dqc_agent.network.select('value')(
        np.repeat(oracle_rep[None], len(candidates), axis=0), candidates
    )
    ob_to_goal_v = dqc_agent.network.select('value')(oracle_rep, goal)

    gamma_to_subgoal = np.log(np.clip(sigmoid(ob_to_subgoal_vs), 1e-6, 1.0)) / np.log(dqc_agent.config['discount'])
    gamma_to_goal    = np.log(np.clip(sigmoid(subgoal_to_goal_vs), 1e-6, 1.0)) / np.log(dqc_agent.config['discount'])
    ob_to_goal       = float(np.log(np.clip(float(sigmoid(ob_to_goal_v)), 1e-6, 1.0)) / np.log(dqc_agent.config['discount']))

    mask = np.asarray(gamma_to_goal).reshape(-1) < ob_to_goal * mult_factor + additive_factor
    pool = candidates[mask] if np.any(mask) else candidates  # fallback: no filter

    novelty_scores = novelty_scorer.score(pool)
    best_idx = int(np.argmax(novelty_scores))
    subgoal = pool[best_idx]
    novelty_scorer.update(subgoal)
    return subgoal, candidates, novelty_scores, mask, rng

In [ ]:
##=========== ROLLOUT WITH COUNT-BASED NOVELTY ===========##

NUM_CANDIDATES  = 128
STEPS_TO_SUBGOAL = 25
NUM_SUBGOALS    = 300
MULT_FACTOR     = 0.9
SEED            = 1000

novelty_scorer = CountBasedNovelty(bin_size=1.0)
rng = jax.random.PRNGKey(SEED)

task_infos = env.unwrapped.task_infos if hasattr(env.unwrapped, 'task_infos') else env.task_infos
task_info = task_infos[0]
goal = np.asarray(task_info['goal_xy'])

ob, _ = env.reset(options=dict(task_id=1))
trajectory = [np.asarray(to_oracle_reps(ob[None], env=env))[0].copy()]
subgoal_history = []

for step in range(NUM_SUBGOALS):
    subgoal, candidates, novelty_scores, mask, rng = select_subgoal_with_novelty(
        ob, goal, novelty_scorer, rng,
        num_candidates=NUM_CANDIDATES,
        mult_factor=MULT_FACTOR,
    )
    subgoal_history.append(subgoal.copy())

    for _ in range(STEPS_TO_SUBGOAL):
        rng, action_rng = jax.random.split(rng)
        action = np.asarray(np.clip(
            dqc_agent.sample_actions(observations=ob, goals=subgoal, seed=action_rng, best_of_n_override=2),
            -1, 1,
        ))
        ob, _, terminated, truncated, _ = env.step(action)
        oracle_rep = np.asarray(to_oracle_reps(ob[None], env=env))[0]
        trajectory.append(oracle_rep.copy())
        if terminated or truncated:
            ob, _ = env.reset(options=dict(task_id=1))

    if step % 50 == 0:
        print(f'step {step:3d}  unique cells: {novelty_scorer.num_unique_cells}  mask_size: {int(np.sum(mask))}')

trajectory = np.stack(trajectory)
print(f'\nDone. Unique cells visited: {novelty_scorer.num_unique_cells}')

In [3]:
print('test')

test
